In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

cust_df = pd.read_csv('../data/train.csv', encoding = 'latin-1')
cust_df.shape

(76020, 371)

In [2]:
cust_df.head(3)

,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.17,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.03,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.77,0


In [3]:
cust_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


# 100명중 4명이 불만족하는 데이터

In [4]:
cust_df['TARGET'].value_counts() # 0: 만족, 1: 불만족
unsatisfied_cnt = cust_df[cust_df['TARGET']==1].TARGET.count()
total_cnt = cust_df.TARGET.count()

print(f'불만족 비율 : {unsatisfied_cnt/total_cnt:.2f}')

불만족 비율 : 0.04


# 최소값, Q1, Q2, Q3가 0인 피쳐가 많다
# 최소값 -999999, 000000인 피처 VAR3 > NAN > 중위값/최빈값 2로 대체

In [5]:
cust_df.describe()

,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
count,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,...,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,7.602000e+04,76020.000000
mean,75964.050723,-1523.199277,33.212865,86.208265,72.363067,119.529632,3.559130,6.472698,0.412946,0.567352,...,7.935824,1.365146,12.215580,8.784074,31.505324,1.858575,76.026165,56.614351,1.172358e+05,0.039569
std,43781.947379,39033.462364,12.956486,1614.757313,339.315831,546.266294,93.155749,153.737066,30.604864,36.513513,...,455.887218,113.959637,783.207399,538.439211,2013.125393,147.786584,4040.337842,2852.579397,1.826646e+05,0.194945
min,1.000000,-999999.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.163750e+03,0.000000
25%,38104.750000,2.000000,23.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.787061e+04,0.000000
50%,76043.000000,2.000000,28.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.064092e+05,0.000000
75%,113748.750000,2.000000,40.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.187563e+05,0.000000
max,151838.000000,238.000000,105.000000,210000.000000,12888.030000,21024.810000,8237.820000,11073.570000,6600.000000,6600.000000,...,50003.880000,20385.720000,138831.630000,91778.730000,438329.220000,24650.010000,681462.900000,397884.300000,2.203474e+07,1.000000


In [6]:
cust_df.var3.value_counts()

 2         74165
 8           138
-999999      116
 9           110
 3           108
           ...  
 231           1
 188           1
 168           1
 135           1
 87            1
Name: var3, Length: 208, dtype: int64

## feature 선택 - id 삭제, var3은 최빈값 2로 대체체

In [16]:
cust_df.drop('ID', axis=1, inplace=True)
cust_df['var3'].replace(-999999, 2, inplace=True)
cust_df.head()

,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,imp_op_var40_ult1,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.170000,0
1,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.030000,0
2,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.770000,0
3,2,37,0.0,195.0,195.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64007.970000,0
4,2,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0


In [17]:
# X,y 데이터 분할
X_features = cust_df.iloc[:, :-1]
y_labels = cust_df.iloc[:, -1]
X_features.shape, y_labels.shape

((76020, 369), (76020,))

In [18]:
# 훈련 / 테스트 분할
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_features, y_labels, test_size=0.2, random_state=0)
# X_train.count()
print(f'학습데이터 shape : {X_train.shape}, 테스트 데이터 shape : {X_test.shape}')

학습데이터 shape : (60816, 369), 테스트 데이터 shape : (15204, 369)


In [19]:
train_cnt = y_train.count()
test_cnt = y_test.count()
print(f'학습 데이터 값의 비율 : {y_train.value_counts()/train_cnt}, 테스트 세트 레이블 값 분포 비율 : {y_test.value_counts()/test_cnt}')

학습 데이터 값의 비율 : 0    0.960964
1    0.039036
Name: TARGET, dtype: float64, 테스트 세트 레이블 값 분포 비율 : 0    0.9583
1    0.0417
Name: TARGET, dtype: float64


In [20]:
# DT 예측기

In [21]:
# RF 예측기

# XGBoost 예측기

## 훈련데이터 > 훈련 * 검증 분할

In [22]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=0)

## XGBosst model

In [23]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

xgb_clf = XGBClassifier(n_estimator = 500, learing_rate = 0.05, random_state = 156)
xgb_clf.fit(X_tr,y_tr, early_stopping_rounds = 100, eval_metric='auc', eval_set =[(X_tr, y_tr), (X_val, y_val)])
xgb_roc_score = roc_auc_score(y_test, xgb_clf.predict_proba(X_test)[:, 1])
xgb_roc_score

[14:26:07] WARNING: D:\bld\xgboost-split_1637426510059\work\src\learner.cc:576: 
Parameters: { "learing_rate", "n_estimator" } might not be used.

  This could be a false alarm, with some parameters getting used by language bindings but
  then being mistakenly passed down to XGBoost core, or some parameter actually being used
  but getting flagged wrongly here. Please open an issue if you find any such cases.


[0]	validation_0-auc:0.82179	validation_1-auc:0.80068
[1]	validation_0-auc:0.83374	validation_1-auc:0.80818
[2]	validation_0-auc:0.83855	validation_1-auc:0.81008
[3]	validation_0-auc:0.84400	validation_1-auc:0.81515
[4]	validation_0-auc:0.84723	validation_1-auc:0.81543
[5]	validation_0-auc:0.85252	validation_1-auc:0.81729
[6]	validation_0-auc:0.85937	validation_1-auc:0.82418
[7]	validation_0-auc:0.86427	validation_1-auc:0.82763
[8]	validation_0-auc:0.86932	validation_1-auc:0.82938
[9]	validation_0-auc:0.87190	validation_1-auc:0.83084
[10]	validation_0-auc:0.87693	validation_1-au

0.8415464125109068

# 최적화 파라미터
n_estimator = 500
learning_rate = 0.15
max_depth = 5

# 하이퍼 오피티 파라미터  최적화화

In [26]:
from hyperopt.pyll.base import scope
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from hyperopt import fmin, tpe, Trials, hp

# 하이퍼파라미터 검색공간
search_space = { 'n_estimator': scope.int(hp.quniform('n_estimator', 50, 300, 10)), 
                'max_depth': scope.int(hp.quniform('max_depth', 3, 10, 1)),
                'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),
                'subsample':hp.uniform('subsample', 0.5, 1.0),
                'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1.0),
                }

In [31]:
from hyperopt import fmin, tpe, Trials, STATUS_OK
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

def objective_func_xgb(params):
    model = XGBClassifier(
        n_estimators=100, # params['n_estimator'],
        max_depth = params['max_depth'],
        learning_rate = params['learning_rate'],
        # subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        random_state=42,
        eval_metric='logloss'
    )
    score_mean = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy').mean()
    return {'loss': -score_mean, 'status': STATUS_OK}

In [32]:
trials = Trials()

best_params = fmin(
    fn=objective_func_xgb,
    space=search_space,
    algo= tpe.suggest,
    max_evals=50,
    trials= trials 
    )

100%|██████████| 50/50 [16:18<00:00, 19.57s/trial, best loss: -0.9610299907133951]


In [36]:
best_params

{'colsample_bytree': 0.5160548964359934,
 'learning_rate': 0.017265107835005757,
 'max_depth': 7.0,
 'n_estimator': 230.0,
 'subsample': 0.8827289415806837}

In [41]:
best_model = XGBClassifier(n_estimators = int(best_params['n_estimator']),
                           learning_rate = best_params['learning_rate'],
                           max_depth = int(best_params['max_depth']),
                           subsample=best_params['subsample'],
                           colsample_bytree = best_params['colsample_bytree'],
                           random_state = 42,
                           eval_metric= 'logloss'
                           )

In [42]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=0)

In [43]:
eval_list = [(X_tr, y_tr), (X_val,y_val)]

best_model.fit(X_tr,y_tr,
                early_stopping_rounds = 100,
                eval_set = eval_list,
                eval_metric = 'auc'
                )

[0]	validation_0-auc:0.72291	validation_1-auc:0.70037
[1]	validation_0-auc:0.76606	validation_1-auc:0.73839
[2]	validation_0-auc:0.77145	validation_1-auc:0.73998
[3]	validation_0-auc:0.77114	validation_1-auc:0.73802
[4]	validation_0-auc:0.77221	validation_1-auc:0.73755
[5]	validation_0-auc:0.77396	validation_1-auc:0.73997
[6]	validation_0-auc:0.81084	validation_1-auc:0.78308
[7]	validation_0-auc:0.81020	validation_1-auc:0.78238
[8]	validation_0-auc:0.81037	validation_1-auc:0.78256
[9]	validation_0-auc:0.81937	validation_1-auc:0.79259
[10]	validation_0-auc:0.81927	validation_1-auc:0.79142
[11]	validation_0-auc:0.82596	validation_1-auc:0.79826
[12]	validation_0-auc:0.82449	validation_1-auc:0.79609
[13]	validation_0-auc:0.82863	validation_1-auc:0.80141
[14]	validation_0-auc:0.82759	validation_1-auc:0.80026
[15]	validation_0-auc:0.82743	validation_1-auc:0.79986
[16]	validation_0-auc:0.82949	validation_1-auc:0.80206
[17]	validation_0-auc:0.82925	validation_1-auc:0.80143
[18]	validation_0-au

XGBClassifier(base_score=0.5, booster='gbtree', colsample_bylevel=1,
              colsample_bynode=1, colsample_bytree=0.5160548964359934,
              enable_categorical=False, eval_metric='logloss', gamma=0,
              gpu_id=-1, importance_type=None, interaction_constraints='',
              learning_rate=0.017265107835005757, max_delta_step=0, max_depth=7,
              min_child_weight=1, missing=nan, monotone_constraints='()',
              n_estimators=230, n_jobs=20, num_parallel_tree=1,
              predictor='auto', random_state=42, reg_alpha=0, reg_lambda=1,
              scale_pos_weight=1, subsample=0.8827289415806837,
              tree_method='exact', validate_parameters=1, verbosity=None)

In [44]:
from sklearn.metrics import roc_auc_score
xgb_roc_score = roc_auc_score(y_test, best_model.predict_proba(X_test)[:,1])
xgb_roc_score

0.8442884237738406

# LGBM으로 돌리기기

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

lgbm_clf = LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=156)

eval_list = [(X_tr, y_tr), (X_val, y_val)]  # 훈련용, 검증용 데이터 평가목록
lgbm_clf.fit(X_tr, y_tr, 
             early_stopping_rounds=100, 
             eval_metric='logloss', 
             eval_set=eval_list, 
             verbose=True)

[1]	training's binary_logloss: 0.159747	valid_1's binary_logloss: 0.161487
[2]	training's binary_logloss: 0.155995	valid_1's binary_logloss: 0.158489
[3]	training's binary_logloss: 0.152875	valid_1's binary_logloss: 0.155925
[4]	training's binary_logloss: 0.150255	valid_1's binary_logloss: 0.153849
[5]	training's binary_logloss: 0.148052	valid_1's binary_logloss: 0.152054
[6]	training's binary_logloss: 0.146074	valid_1's binary_logloss: 0.1505
[7]	training's binary_logloss: 0.144325	valid_1's binary_logloss: 0.149162
[8]	training's binary_logloss: 0.142724	valid_1's binary_logloss: 0.14795
[9]	training's binary_logloss: 0.141297	valid_1's binary_logloss: 0.146862
[10]	training's binary_logloss: 0.139983	valid_1's binary_logloss: 0.145886
[11]	training's binary_logloss: 0.138763	valid_1's binary_logloss: 0.144993
[12]	training's binary_logloss: 0.137652	valid_1's binary_logloss: 0.14414
[13]	training's binary_logloss: 0.136616	valid_1's binary_logloss: 0.1434
[14]	training's binary_logl

LGBMClassifier(learning_rate=0.05, n_estimators=500, random_state=156)

In [65]:
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score
def get_clf_eval(y_test, pred=None, pred_probs=None):
    confusion = confusion_matrix(y_test, pred)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    roc_auc = roc_auc_score(y_test, pred_probs)  # 수정된 부분

    print(confusion)
    print('*' * 20)
    print(f"Accuracy: {accuracy}, Precision: {precision}, Recall: {recall}, F1-Score: {f1}, ROC-AUC: {roc_auc}")
    return roc_auc  # AUC 값을 반환하도록 수정

In [66]:
preds = lgbm_clf.predict(X_test)
pred_proba = lgbm_clf.predict_proba(X_test)[:, 1]

# 성능 평가
lgbm_roc_auc = get_clf_eval(y_test, preds, pred_proba)

[[14565     5]
 [  630     4]]
********************
Accuracy: 0.9582346750855039, Precision: 0.4444444444444444, Recall: 0.006309148264984227, F1-Score: 0.012441679626749611, ROC-AUC: 0.8403631765717119


# XGBM vs LGBM

1. auc 기준 평가 결과
2. 각각의 파라미터 값

In [67]:
# AUC 결과 출력
print(f"XGBoost ROC-AUC: {xgb_roc_score:.4f}")

XGBoost ROC-AUC: 0.8443


In [71]:
print(f"LGBM ROC-AUC : {lgbm_roc_auc:.4f}")

LGBM ROC-AUC : 0.8404


In [69]:
# 모델 파라미터 출력
xgb_params = xgb_clf.get_params()
xgb_params

{'objective': 'binary:logistic',
 'use_label_encoder': True,
 'base_score': 0.5,
 'booster': 'gbtree',
 'colsample_bylevel': 1,
 'colsample_bynode': 1,
 'colsample_bytree': 1,
 'enable_categorical': False,
 'gamma': 0,
 'gpu_id': -1,
 'importance_type': None,
 'interaction_constraints': '',
 'learning_rate': 0.300000012,
 'max_delta_step': 0,
 'max_depth': 6,
 'min_child_weight': 1,
 'missing': nan,
 'monotone_constraints': '()',
 'n_estimators': 100,
 'n_jobs': 20,
 'num_parallel_tree': 1,
 'predictor': 'auto',
 'random_state': 156,
 'reg_alpha': 0,
 'reg_lambda': 1,
 'scale_pos_weight': 1,
 'subsample': 1,
 'tree_method': 'exact',
 'validate_parameters': 1,
 'verbosity': None,
 'n_estimator': 500,
 'learing_rate': 0.05}

In [70]:
# 모델 파라미터 출력
lgbm_params = lgbm_clf.get_params()
lgbm_params

{'boosting_type': 'gbdt',
 'class_weight': None,
 'colsample_bytree': 1.0,
 'importance_type': 'split',
 'learning_rate': 0.05,
 'max_depth': -1,
 'min_child_samples': 20,
 'min_child_weight': 0.001,
 'min_split_gain': 0.0,
 'n_estimators': 500,
 'n_jobs': -1,
 'num_leaves': 31,
 'objective': None,
 'random_state': 156,
 'reg_alpha': 0.0,
 'reg_lambda': 0.0,
 'silent': 'warn',
 'subsample': 1.0,
 'subsample_for_bin': 200000,
 'subsample_freq': 0}